In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point
import os

DATA_PATH = "../data"
OUTPUT_PATH = "../outputs"

os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
admin = gpd.read_file(f"{DATA_PATH}/admin_boundaries.geojson")
infra = gpd.read_file(f"{DATA_PATH}/infrastructure.geojson")
lulc = gpd.read_file(f"{DATA_PATH}/lulc.geojson")

In [ ]:
infra = infra.dropna(subset=["geometry"])
infra = infra[infra.geometry.type == "Point"]

#
infra = infra[infra["amenity"].isin(["hospital", "fire_station"])]

In [ ]:
centroid = infra.geometry.unary_union.centroid
utm_zone = int((centroid.x + 180) / 6) + 1
utm_crs = f"EPSG:326{utm_zone}"

admin = admin.to_crs(utm_crs)
infra = infra.to_crs(utm_crs)
lulc = lulc.to_crs(utm_crs)

In [ ]:
infra_join = gpd.sjoin(
    infra,
    admin,
    predicate="within",
    how="left",
    lsuffix="infra",
    rsuffix="admin"
)

possible_name_cols = ["name", "NAME", "district", "District", "ADM2_EN", "admin", "ward"]

for col in possible_name_cols:
    if col in infra_join.columns:
        district_col = col
        break

# district_counts = infra_join.groupby(district_col).size().reset_index(name="asset_count")
# district_counts.sort_values("asset_count", ascending=False)

# district_counts = infra_join.groupby("name").size().reset_index(name="asset_count")
# district_counts = district_counts.sort_values("asset_count", ascending=False)

district_counts = (
    infra_join.groupby("name_admin")
    .size()
    .reset_index(name="asset_count")
    .sort_values("asset_count", ascending=False)
)

district_counts

In [ ]:
hospitals = infra[infra["amenity"] == "hospital"].copy()

hospitals["buffer_1km"] = hospitals.geometry.buffer(1000)
buffer_1km = gpd.GeoDataFrame(geometry=hospitals["buffer_1km"], crs=utm_crs)

overlay = gpd.overlay(lulc, buffer_1km, how="intersection")
overlay["area"] = overlay.geometry.area

landuse_summary = overlay.groupby("landuse")["area"].sum()
landuse_summary = (landuse_summary / landuse_summary.sum()) * 100

landuse_summary.sort_values(ascending=False)

In [ ]:
res = lulc[lulc["landuse"] == "Residential"].copy()
res = res.to_crs(utm_crs)

fires = infra[infra["amenity"] == "fire_station"].copy()
fires = fires.to_crs(utm_crs)

res["centroid"] = res.geometry.centroid

res_points = gpd.GeoDataFrame(
    res.drop(columns="geometry"),
    geometry="centroid",
    crs=utm_crs
).reset_index(drop=True)

fires = fires.reset_index(drop=True)

def nearest_fire_distance(point, fire_gdf):
    return fire_gdf.distance(point).min()

res_points["distance_m"] = res_points.geometry.apply(
    lambda x: nearest_fire_distance(x, fires)
).astype(float)

res_points["risk"] = np.where(
    res_points["distance_m"] > 10000,
    "High Risk",
    "Secure"
)

res_points[["distance_m", "risk"]].head()

In [ ]:
os.makedirs(f"{OUTPUT_PATH}/maps", exist_ok=True)

def clean_filename(name):
    return re.sub(r'[\/:*?"<>|]', '_', str(name))

admin_poly = admin[admin.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()

infra = infra.to_crs(admin_poly.crs)

for _, row in admin_poly.iterrows():

    district_name = clean_filename(row["name"])
    geom = row.geometry

    mask = gpd.GeoSeries([geom], crs=admin_poly.crs)

    clipped = gpd.clip(infra, mask)

    if clipped.empty:
        continue

    fig, ax = plt.subplots(figsize=(10, 10))

    gpd.GeoSeries([geom]).boundary.plot(ax=ax, color="black", linewidth=2)

    clipped.plot(
        ax=ax,
        column="amenity",
        categorical=True,
        legend=True,
        markersize=20
    )

    ax.set_title(f"District Assessment: {district_name}")
    ax.axis("off")

    save_path = f"{OUTPUT_PATH}/maps/District_Assessment_{district_name}.pdf"

    plt.savefig(save_path, bbox_inches="tight")
    plt.close()

In [ ]:
print("District counts:")
display(district_counts)

print("Risk summary:")
display(res_points["risk"].value_counts())